In [6]:
#@title Mount Goole Drive

import os
import sys
from pathlib import Path

root_path = "/content/drive/MyDrive/MSc/Flood-Mapping"  #@param {type:"string", multiline:true}

mount_drive = True  #@param {type:"boolean"}

clone_repo = False  #@param {type:"boolean"}

if clone_repo and mount_drive:

    from google.colab import drive
    drive.mount("/content/drive")

    root_path = os.path.join(root_path, "Flood-Mapping")

    !git clone https://github.com/TAX2310/Flood-Mapping.git $root_path

    sys.path.append(os.path.join(root_path))

    from S1.s1_config import S1_CFG
    cfg = S1_CFG()

    cfg.ROOT = Path(root_path)

elif not clone_repo and mount_drive:
    from google.colab import drive
    drive.mount("/content/drive")

    sys.path.append(os.path.join(root_path))

    from S1.s1_config import S1_CFG
    cfg = S1_CFG()

    cfg.ROOT = Path(root_path)

elif clone_repo and not mount_drive:
    root_path = "Flood-Mapping"

    !git clone https://github.com/TAX2310/Flood-Mapping.git

    sys.path.append(root_path)

    from S1.s1_config import S1_CFG
    cfg = S1_CFG()

    cfg.ROOT = Path(root_path)

else:
    from S1.s1_config import S1_CFG
    cfg = S1_CFG()

    sys.path.append(os.path.join(cfg.ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
from S1.data.dataloader import *

In [8]:
train_loader, val_loader, test_loader = make_dataloaders(cfg)

batch = next(iter(train_loader))
print(batch["image"].shape)   # [B, 2, 128, 128]
print(batch["mask"].shape)    # [B, 128, 128]
print(batch["image"].dtype, batch["mask"].dtype)
print(batch["id"][:3])

torch.Size([16, 2, 128, 128])
torch.Size([16, 128, 128])
torch.float32 torch.int64
['EMSR429_AOI03_05_06_1_2', 'EMSR517_AOI05_11_08_2_2', 'EMSR352_AOI01_090_024_2_2']


In [9]:

train_events = train_loader.dataset.get_events()
val_events   = val_loader.dataset.get_events()
test_events  = test_loader.dataset.get_events()

print("Train events:", len(train_events))
print("Val events:", len(val_events))
print("Test events:", len(test_events))

Train events: 37
Val events: 5
Test events: 5


In [10]:
train_events = set(train_loader.dataset.get_events())
val_events = set(val_loader.dataset.get_events())
test_events = set(test_loader.dataset.get_events())

print(train_events & val_events)
print(train_events & test_events)
print(val_events & test_events)

set()
set()
set()


In [13]:
!pip install segmentation-models-pytorch

import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="resnet34",        # ResNet34 encoder
    encoder_weights=None,           # None for SAR (not RGB ImageNet)
    in_channels=2,                  # Your S1 input (VV, VH)
    classes=1                       # Binary segmentation output
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 3.3 MB/s eta 0:00:00


In [15]:
from segmentation_models_pytorch.losses import DiceLoss
import torch.nn as nn

bce = nn.BCEWithLogitsLoss()
dice = DiceLoss(mode="binary")

def loss_fn(pred, target):
    return 0.5 * bce(pred, target) + 0.5 * dice(pred, target)

In [16]:
import torch.optim as optim

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)

In [17]:
print(model)

Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(2, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track